In [3]:
# imports

import pandas as pd
import numpy as np
import json
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import polars as pl
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [24]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, entailed_status, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": entailed_status,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, entailed_status in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": entailed_status,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [25]:
# Load in table dataframe

df_tables = (
    pl.read_csv(
        "../tabfact_data/tables/*.csv",
        has_header=False,
        separator="\1", # Non-existent delimiter so full line/file stays intact
        new_columns=["table_content"],
        include_file_paths="file_name",
    )
    .group_by("file_name", maintain_order=True) # Group all the table lines together 
    .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
    .to_pandas()
)

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

df_tables.shape

(16573, 2)

In [27]:
# Load in data frames

df_full_cleaned = load_and_convert_data_to_dataframe("../tabfact_data/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1_training = load_and_convert_data_to_dataframe("../tabfact_data/r1_training_all.json")

df_r2_training = load_and_convert_data_to_dataframe("../tabfact_data/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe("../tabfact_data/train_examples.json")

df_test = load_and_convert_data_to_dataframe("../tabfact_data/test_examples.json")

df_validation = load_and_convert_data_to_dataframe("../tabfact_data/val_examples.json")


In [55]:
# Defining column labelled rows
print(df_tables.shape)

df_tables["headers"] = df_tables["table_content"].apply(lambda x: x.split("\n")[0].split("#")) # Get the first line of the table content as headers

df_tables["rows"] = df_tables["table_content"].apply(lambda x: [row.split("#") for row in x.split("\n")[1:]]) # Get the rest of the lines as rows

df_tables["column_labelled_rows"] = df_tables.apply(
    lambda row: "\n".join(
        f"{", ".join(f"{h}: {v}" for h, v in zip(row["headers"], r))}"
        for r in row["rows"]
    ),
    axis=1,
)

print(df_tables["column_labelled_rows"].iloc[0])

(16573, 5)
aircraft: robinson r - 22, description: light utility helicopter, max gross weight: 1370 lb (635 kg), total disk area: 497 ft square (46.2 m square), max disk loading: 2.6 lb / ft square (14 kg / m square)
aircraft: bell 206b3 jetranger, description: turboshaft utility helicopter, max gross weight: 3200 lb (1451 kg), total disk area: 872 ft square (81.1 m square), max disk loading: 3.7 lb / ft square (18 kg / m square)
aircraft: ch - 47d chinook, description: tandem rotor helicopter, max gross weight: 50000 lb (22680 kg), total disk area: 5655 ft square (526 m square), max disk loading: 8.8 lb / ft square (43 kg / m square)
aircraft: mil mi - 26, description: heavy - lift helicopter, max gross weight: 123500 lb (56000 kg), total disk area: 8495 ft square (789 m square), max disk loading: 14.5 lb / ft square (71 kg / m square)
aircraft: ch - 53e super stallion, description: heavy - lift helicopter, max gross weight: 73500 lb (33300 kg), total disk area: 4900 ft square (460 m 

In [75]:
def get_table_claim_df(df, table_df_col, table_before_claim=True, table_prefix="TABLE: ", claim_prefix="CLAIM: ", table_claim_separator=" | "):
    df_table_claim = pd.merge(
        df_tables[["file_name", table_df_col]],
        df[["table_id", "statement", "label"]],
        left_on="file_name",
        right_on="table_id",
        how="inner"
    ).reset_index(drop=True)

    if table_before_claim:
        df_table_claim["X_val"] = table_prefix + df_table_claim[table_df_col] + table_claim_separator + claim_prefix + df_table_claim["statement"]
    else:
        df_table_claim["X_val"] = claim_prefix + df_table_claim["statement"] + table_claim_separator + table_prefix + df_table_claim[table_df_col]

    return df_table_claim.copy()

# Get training table-claim corpus for tfidf vectorizer
df_train_table_claim = get_table_claim_df(df_training, "column_labelled_rows", table_prefix="", claim_prefix="", table_claim_separator="\n", table_before_claim=False)
train_table_claim_corpus = df_train_table_claim["X_val"].to_numpy()

# Get training table-claim embeddings and truth labels
vectorizer = TfidfVectorizer(max_features=30000)
X_train = vectorizer.fit_transform(train_table_claim_corpus)
y_train = df_train_table_claim["label"]

# # Get validation table-claim corpus for tfidf vectorizer
# df_validation_table_claim = get_table_claim_df(df_validation, "column_labelled_rows", table_prefix="", claim_prefix="", table_claim_separator="\n", table_before_claim=False)
# validation_table_claim_corpus = df_validation_table_claim["X_val"].to_numpy()

# # Get validation table-claim embeddings and truth labels
# X_validation = vectorizer.transform(validation_table_claim_corpus)
# y_validation = df_validation_table_claim["label"]

# Get test table-claim corpus for tfidf vectorizer
df_test_table_claim = get_table_claim_df(df_test, "column_labelled_rows", table_prefix="", claim_prefix="", table_claim_separator="\n", table_before_claim=False)
test_table_claim_corpus = df_test_table_claim["X_val"].to_numpy()

# Get test table-claim embeddings and truth labels
X_test = vectorizer.transform(test_table_claim_corpus)
y_test = df_test_table_claim["label"]

In [76]:
df_train_table_claim["X_val"].iloc[0]

'robinson r - 22 be the aircraft with the lowest max gross weight\naircraft: robinson r - 22, description: light utility helicopter, max gross weight: 1370 lb (635 kg), total disk area: 497 ft square (46.2 m square), max disk loading: 2.6 lb / ft square (14 kg / m square)\naircraft: bell 206b3 jetranger, description: turboshaft utility helicopter, max gross weight: 3200 lb (1451 kg), total disk area: 872 ft square (81.1 m square), max disk loading: 3.7 lb / ft square (18 kg / m square)\naircraft: ch - 47d chinook, description: tandem rotor helicopter, max gross weight: 50000 lb (22680 kg), total disk area: 5655 ft square (526 m square), max disk loading: 8.8 lb / ft square (43 kg / m square)\naircraft: mil mi - 26, description: heavy - lift helicopter, max gross weight: 123500 lb (56000 kg), total disk area: 8495 ft square (789 m square), max disk loading: 14.5 lb / ft square (71 kg / m square)\naircraft: ch - 53e super stallion, description: heavy - lift helicopter, max gross weight: 

In [77]:
# Performing logistic regression

logistic_regression_model = LogisticRegression(solver="lbfgs", max_iter=10000, random_state=0).fit(X_train, y_train)
y_test_pred = logistic_regression_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_test_pred)}")
print(f"Precision: {precision_score(y_test, y_test_pred)}")
print(f"Recall: {recall_score(y_test, y_test_pred)}")
print(f"F1-Score: {f1_score(y_test, y_test_pred)}")

Accuracy: 0.5171766178887237
Precision: 0.5122466621842282
Recall: 0.8300389105058366
F1-Score: 0.6335234022333096
